In [6]:
# =============================================================================
# Notebook 1 — EDA / Exploración de datos (F1, jolpica-f1)
# Fase: vista previa + esquema, nulos por key aplanada y validación de PKs.
# Cada bloque "# %%" corresponde a una celda del notebook.
# =============================================================================

# %% ------------------------------------------------------------------------
# 1. Conexión Spark
# ----------------------------------------------------------------------------
from pyspark.sql import SparkSession
# from pyspark.sql.functions import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, ArrayType

spark = SparkSession.builder \
    .appName("F1_EDA_Refinamiento") \
    .enableHiveSupport() \
    .getOrCreate()

# Reducir ruido de logs para que la demo se vea limpia
spark.sparkContext.setLogLevel("WARN")

LND_PATH = "ort/LND"
RFN_PATH = "ort/RFN"

# %% ------------------------------------------------------------------------
# 2. Carga de tablas desde /LND
# ----------------------------------------------------------------------------
TABLAS_LND = [
    "seasons",
    "drivers",
    "constructors",
    "circuits",
    "status",
    "races",
    "results",
    "constructor_standings",
]

def cargar_tabla(nombre):
    """Lee el JSON multilínea de una tabla desde /LND."""
    return spark.read.option("multiLine", True).json(f"{LND_PATH}/{nombre}.json")

# Un solo dict con todos los DataFrames crudos, indexado por nombre de tabla
DATAFRAMES = {nombre: cargar_tabla(nombre) for nombre in TABLAS_LND}

# %% ------------------------------------------------------------------------
# 3. Helpers generales
# ----------------------------------------------------------------------------
def _q(nombre):
    """Escapa un nombre de columna con backticks (necesario cuando el alias contiene puntos)."""
    return f"`{nombre}`"

def perfil(df, nombre, show_data=True):
    """Vista previa + cantidad/nombre de columnas + esquema."""
    print(f"===== Tabla: {nombre} =====")
    print(f"Filas: {df.count()}   |   Columnas: {len(df.columns)}")
    print(f"Nombres de columnas: {df.columns}\n")
    print("Esquema:")
    df.printSchema()
    if show_data:
        print("Primeras filas:")
        df.show(5, truncate=False)

# %% ------------------------------------------------------------------------
# 4. Aplanado recursivo (structs + arrays -> columnas hoja)
# ----------------------------------------------------------------------------
def aplanar(df, sep="."):
    """Aplana el DataFrame hasta dejar solo columnas hoja (tipos simples).
    - struct  -> una columna por campo, con alias 'padre.hijo'
    - array   -> explode_outer (una fila por elemento; conserva filas con array nulo/vacío)
    ⚠️ Los arrays cambian el grano: la cantidad de filas del resultado no es la original.
    """
    while True:
        complejo = next(
            ((f.name, f.dataType) for f in df.schema.fields
             if isinstance(f.dataType, (StructType, ArrayType))),
            None
        )
        if complejo is None:
            return df
        nombre, tipo = complejo
        if isinstance(tipo, ArrayType):
            df = df.withColumn(nombre, F.explode_outer(F.col(_q(nombre))))
        else:  # StructType: reemplazar la columna por sus campos, respetando el orden
            seleccion = []
            for c in df.columns:
                if c != nombre:
                    seleccion.append(F.col(_q(c)))
                else:
                    for hijo in tipo.fields:
                        seleccion.append(
                            F.col(_q(nombre))[hijo.name].alias(f"{nombre}{sep}{hijo.name}")
                        )
            df = df.select(seleccion)

# %% ------------------------------------------------------------------------
# 5. Nulos
# ----------------------------------------------------------------------------
def reporte_nulos(df, etiqueta_fila="filas"):
    """Reporte de nulos/vacíos por columna sobre un DataFrame ya plano."""
    total = df.count()
    expr_nulos  = [F.count(F.when(F.col(_q(c)).isNull(), True)).alias(c) for c in df.columns]
    expr_vacios = [F.count(F.when(F.col(_q(c)).isNotNull() &
                                  (F.trim(F.col(_q(c)).cast("string")) == ""), True)).alias(c)
                   for c in df.columns]
    fila_nulos  = df.select(expr_nulos).first()
    fila_vacios = df.select(expr_vacios).first()

    ancho = max(len(c) for c in df.columns) + 2
    print(f"Total de {etiqueta_fila}: {total}\n")
    print(f"{'Columna':{ancho}s} | {'Nulos':>7s} | {'Vacíos':>7s} | {'Con dato':>8s} | {'% nulos':>7s}")
    print("-" * (ancho + 42))
    for c in df.columns:
        n, v = fila_nulos[c], fila_vacios[c]
        pct = 100 * n / total if total else 0
        print(f"{c:{ancho}s} | {n:7d} | {v:7d} | {total - n - v:8d} | {pct:6.1f}%")

def nulos_completo(df, nombre):
    """Aplana la tabla (structs + arrays) y reporta nulos/vacíos por cada key hoja."""
    print(f"===== Nulos: {nombre} =====")
    df_plano = aplanar(df).cache()
    reporte_nulos(df_plano, etiqueta_fila="filas (post-aplanado)")
    return df_plano  # se devuelve por si se quiere reutilizar sin recalcular

# %% ------------------------------------------------------------------------
# 6. Validación de PK
# ----------------------------------------------------------------------------
def pk_unica(df, pk):
    """Verifica unicidad de una PK (simple o compuesta) sobre un DataFrame plano."""
    pk = pk if isinstance(pk, list) else [pk]
    total = df.count()
    distintos = df.select([F.col(_q(c)) for c in pk]).distinct().count()
    print(f"PK {pk}: filas={total}, combinaciones distintas={distintos}, es única = {total == distintos}")
    dups = (df.groupBy([F.col(_q(c)) for c in pk])
              .count()
              .filter(F.col("count") > 1)
              .orderBy(F.col("count").desc()))
    if dups.limit(1).count() > 0:
        print("Combinaciones duplicadas de la PK:")
        dups.show(20, truncate=False)

# %% ------------------------------------------------------------------------
# 7. Configuración de PKs por tabla (nombres post-aplanado)
# ----------------------------------------------------------------------------
TABLAS = {
    "seasons":      [["season"]],
    "drivers":      [["driverId"]],
    "constructors": [["constructorId"]],
    "circuits":     [["circuitId"]],
    "status":       [["statusId"]],
    "races":        [["season", "round"]],
    "results": [
        # PK confirmada, base del resultId sintético
        ["season", "round", "Results.Driver.driverId", "Results.number"],
    ],
    "constructor_standings": [
        ["season", "round", "ConstructorStandings.Constructor.constructorId"],
    ],
}

# %% ------------------------------------------------------------------------
# 8. Orquestador
# ----------------------------------------------------------------------------
def explorar(nombre):
    """Corre el análisis de nulos y las validaciones de PK candidatas de una tabla."""
    df_plano = nulos_completo(DATAFRAMES[nombre], nombre)
    print()
    for pk in TABLAS[nombre]:
        pk_unica(df_plano, pk)
    df_plano.unpersist()
    print()


In [2]:
# %% ------------------------------------------------------------------------
# 9. Ejecución
# ----------------------------------------------------------------------------
# 9.1 Vista previa y esquema de cada colección
for nombre, df in DATAFRAMES.items():
    perfil(df, f"{nombre} (LND)")
    print()

===== Tabla: seasons (LND) =====
Filas: 77   |   Columnas: 2
Nombres de columnas: ['season', 'url']

Esquema:
root
 |-- season: string (nullable = true)
 |-- url: string (nullable = true)

Primeras filas:
+------+-----------------------------------------------------+
|season|url                                                  |
+------+-----------------------------------------------------+
|1950  |https://en.wikipedia.org/wiki/1950_Formula_One_season|
|1951  |https://en.wikipedia.org/wiki/1951_Formula_One_season|
|1952  |https://en.wikipedia.org/wiki/1952_Formula_One_season|
|1953  |https://en.wikipedia.org/wiki/1953_Formula_One_season|
|1954  |https://en.wikipedia.org/wiki/1954_Formula_One_season|
+------+-----------------------------------------------------+
only showing top 5 rows


===== Tabla: drivers (LND) =====
Filas: 881   |   Columnas: 8
Nombres de columnas: ['code', 'dateOfBirth', 'driverId', 'familyName', 'givenName', 'nationality', 'permanentNumber', 'url']

Esquema:
root


+---------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# %%
# 9.2 Nulos y PKs de todas las tablas
for nombre in TABLAS:
    explorar(nombre)

# %%
# (Alternativa: correr una tabla puntual)
# explorar("results")

===== Nulos: seasons =====
Total de filas (post-aplanado): 77

Columna  |   Nulos |  Vacíos | Con dato | % nulos
--------------------------------------------------
season   |       0 |       0 |       77 |    0.0%
url      |       0 |       0 |       77 |    0.0%

PK ['season']: filas=77, combinaciones distintas=77, es única = True

===== Nulos: drivers =====
Total de filas (post-aplanado): 881

Columna           |   Nulos |  Vacíos | Con dato | % nulos
-----------------------------------------------------------
code              |     774 |       0 |      107 |   87.9%
dateOfBirth       |      16 |       0 |      865 |    1.8%
driverId          |       0 |       0 |      881 |    0.0%
familyName        |       0 |       0 |      881 |    0.0%
givenName         |       0 |       0 |      881 |    0.0%
nationality       |      16 |       0 |      865 |    1.8%
permanentNumber   |     818 |       0 |       63 |   92.8%
url               |      16 |       0 |      865 |    1.8%

PK ['driv

Total de filas (post-aplanado): 26071

Columna                                 |   Nulos |  Vacíos | Con dato | % nulos
---------------------------------------------------------------------------------
Circuit.Location.country                |       0 |       0 |    26071 |    0.0%
Circuit.Location.lat                    |       0 |       0 |    26071 |    0.0%
Circuit.Location.locality               |       0 |       0 |    26071 |    0.0%
Circuit.Location.long                   |       0 |       0 |    26071 |    0.0%
Circuit.circuitId                       |       0 |       0 |    26071 |    0.0%
Circuit.circuitName                     |       0 |       0 |    26071 |    0.0%
Circuit.url                             |       0 |       0 |    26071 |    0.0%
Results.Constructor.constructorId       |       0 |       0 |    26071 |    0.0%
Results.Constructor.name                |       0 |       0 |    26071 |    0.0%
Results.Constructor.nationality         |       0 |       0 |    2607

In [103]:
# ------------------------------------------- SEASONS -------------------------------------------

df_seasons_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/seasons.json")
perfil(df_seasons_raw, "seasons (LND)")

===== Tabla: seasons (LND) =====
Filas: 77   |   Columnas: 2
Nombres de columnas: ['season', 'url']

Esquema:
root
 |-- season: string (nullable = true)
 |-- url: string (nullable = true)

Primeras filas:
+------+-----------------------------------------------------+
|season|url                                                  |
+------+-----------------------------------------------------+
|1950  |https://en.wikipedia.org/wiki/1950_Formula_One_season|
|1951  |https://en.wikipedia.org/wiki/1951_Formula_One_season|
|1952  |https://en.wikipedia.org/wiki/1952_Formula_One_season|
|1953  |https://en.wikipedia.org/wiki/1953_Formula_One_season|
|1954  |https://en.wikipedia.org/wiki/1954_Formula_One_season|
+------+-----------------------------------------------------+
only showing top 5 rows



In [7]:
nulos(df_seasons_raw)
pk_unica(df_seasons_raw, "season")

Nulos / vacíos por columna:
+------+---+
|season|url|
+------+---+
|0     |0  |
+------+---+

PK ['season']: filas=77, combinaciones distintas=77, es única = True


In [8]:
# ------------------------------------------- DRIVERS -------------------------------------------

df_drivers_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/drivers.json")
perfil(df_drivers_raw, "drivers (LND)")

===== Tabla: drivers (LND) =====
Filas: 881   |   Columnas: 8
Nombres de columnas: ['code', 'dateOfBirth', 'driverId', 'familyName', 'givenName', 'nationality', 'permanentNumber', 'url']

Esquema:
root
 |-- code: string (nullable = true)
 |-- dateOfBirth: string (nullable = true)
 |-- driverId: string (nullable = true)
 |-- familyName: string (nullable = true)
 |-- givenName: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- permanentNumber: string (nullable = true)
 |-- url: string (nullable = true)

Primeras filas:
+----+-----------+---------+----------+---------+-----------+---------------+----------------------------------------------+
|code|dateOfBirth|driverId |familyName|givenName|nationality|permanentNumber|url                                           |
+----+-----------+---------+----------+---------+-----------+---------------+----------------------------------------------+
|null|1932-07-10 |abate    |Abate     |Carlo    |Italian    |null           |ht

In [9]:
nulos(df_drivers_raw)
pk_unica(df_drivers_raw, "driverId")

Nulos / vacíos por columna:
+----+-----------+--------+----------+---------+-----------+---------------+---+
|code|dateOfBirth|driverId|familyName|givenName|nationality|permanentNumber|url|
+----+-----------+--------+----------+---------+-----------+---------------+---+
|774 |16         |0       |0         |0        |16         |818            |16 |
+----+-----------+--------+----------+---------+-----------+---------------+---+

PK ['driverId']: filas=881, combinaciones distintas=881, es única = True


In [10]:
# ------------------------------------------- CONSTRUCTORS -------------------------------------------

df_constructors_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/constructors.json")
perfil(df_constructors_raw, "constructors (LND)")

===== Tabla: constructors (LND) =====
Filas: 214   |   Columnas: 4
Nombres de columnas: ['constructorId', 'name', 'nationality', 'url']

Esquema:
root
 |-- constructorId: string (nullable = true)
 |-- name: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- url: string (nullable = true)

Primeras filas:
+-------------+----------+-----------+------------------------------------------------------------------+
|constructorId|name      |nationality|url                                                               |
+-------------+----------+-----------+------------------------------------------------------------------+
|adams        |Adams     |American   |https://en.wikipedia.org/wiki/Adams_(constructor)                 |
|afm          |AFM       |German     |https://en.wikipedia.org/wiki/Alex_von_Falkenhausen_Motorenbau    |
|ags          |AGS       |French     |https://en.wikipedia.org/wiki/Automobiles_Gonfaronnaises_Sportives|
|alfa         |Alfa Romeo|Swiss      

In [11]:
nulos(df_constructors_raw)
pk_unica(df_constructors_raw, "constructorId")

Nulos / vacíos por columna:
+-------------+----+-----------+---+
|constructorId|name|nationality|url|
+-------------+----+-----------+---+
|0            |0   |0          |0  |
+-------------+----+-----------+---+

PK ['constructorId']: filas=214, combinaciones distintas=214, es única = True


In [12]:
# ------------------------------------------- STATUS -------------------------------------------

df_status_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/status.json")
perfil(df_status_raw, "status (LND)")

===== Tabla: status (LND) =====
Filas: 136   |   Columnas: 3
Nombres de columnas: ['count', 'status', 'statusId']

Esquema:
root
 |-- count: string (nullable = true)
 |-- status: string (nullable = true)
 |-- statusId: string (nullable = true)

Primeras filas:
+-----+--------+--------+
|count|status  |statusId|
+-----+--------+--------+
|8093 |Finished|1       |
|3850 |+1 Lap  |11      |
|2011 |Engine  |5       |
|1593 |+2 Laps |12      |
|1047 |Accident|3       |
+-----+--------+--------+
only showing top 5 rows



In [13]:
nulos(df_status_raw)
pk_unica(df_status_raw, "statusId")

Nulos / vacíos por columna:
+-----+------+--------+
|count|status|statusId|
+-----+------+--------+
|0    |0     |0       |
+-----+------+--------+

PK ['statusId']: filas=136, combinaciones distintas=136, es única = True


In [14]:
# ------------------------------------------- CIRCUITS -------------------------------------------

df_circuits_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/circuits.json")
perfil(df_circuits_raw, "circuits (LND)")

===== Tabla: circuits (LND) =====
Filas: 78   |   Columnas: 4
Nombres de columnas: ['Location', 'circuitId', 'circuitName', 'url']

Esquema:
root
 |-- Location: struct (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- lat: string (nullable = true)
 |    |-- locality: string (nullable = true)
 |    |-- long: string (nullable = true)
 |-- circuitId: string (nullable = true)
 |-- circuitName: string (nullable = true)
 |-- url: string (nullable = true)

Primeras filas:
+-----------------------------------------+-----------+------------------------------+----------------------------------------------------------+
|Location                                 |circuitId  |circuitName                   |url                                                       |
+-----------------------------------------+-----------+------------------------------+----------------------------------------------------------+
|{Australia, -34.9272, Adelaide, 138.617} |adelaide   |Adelaide Street

In [15]:
df_circuits_raw.select("circuitId", "Location.*").show(5, truncate=False)

+-----------+---------+--------+----------+--------+
|circuitId  |country  |lat     |locality  |long    |
+-----------+---------+--------+----------+--------+
|adelaide   |Australia|-34.9272|Adelaide  |138.617 |
|ain-diab   |Morocco  |33.5786 |Casablanca|-7.6875 |
|aintree    |UK       |53.4769 |Liverpool |-2.94056|
|albert_park|Australia|-37.8497|Melbourne |144.968 |
|americas   |USA      |30.1328 |Austin    |-97.6411|
+-----------+---------+--------+----------+--------+
only showing top 5 rows



In [16]:
nulos(df_circuits_raw)
pk_unica(df_circuits_raw, "circuitId")

Nulos / vacíos por columna:
+---------+-----------+---+
|circuitId|circuitName|url|
+---------+-----------+---+
|0        |0          |0  |
+---------+-----------+---+

PK ['circuitId']: filas=78, combinaciones distintas=78, es única = True


In [17]:
# ------------------------------------------- RACES -------------------------------------------

df_races_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/races.json")
perfil(df_races_raw, "races (LND)")

===== Tabla: races (LND) =====
Filas: 1171   |   Columnas: 14
Nombres de columnas: ['Circuit', 'FirstPractice', 'Qualifying', 'SecondPractice', 'Sprint', 'SprintQualifying', 'SprintShootout', 'ThirdPractice', 'date', 'raceName', 'round', 'season', 'time', 'url']

Esquema:
root
 |-- Circuit: struct (nullable = true)
 |    |-- Location: struct (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- lat: string (nullable = true)
 |    |    |-- locality: string (nullable = true)
 |    |    |-- long: string (nullable = true)
 |    |-- circuitId: string (nullable = true)
 |    |-- circuitName: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- FirstPractice: struct (nullable = true)
 |    |-- date: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- Qualifying: struct (nullable = true)
 |    |-- date: string (nullable = true)
 |    |-- time: string (nullable = true)
 |-- SecondPractice: struct (nullable = true)
 |    |-- date: s

In [18]:
df_races_raw.select("Circuit.*").show(5, truncate=False)

+---------------------------------------+------------+----------------------------+----------------------------------------------------------+
|Location                               |circuitId   |circuitName                 |url                                                       |
+---------------------------------------+------------+----------------------------+----------------------------------------------------------+
|{UK, 52.0786, Silverstone, -1.01694}   |silverstone |Silverstone Circuit         |https://en.wikipedia.org/wiki/Silverstone_Circuit         |
|{Monaco, 43.7347, Monte Carlo, 7.42056}|monaco      |Circuit de Monaco           |https://en.wikipedia.org/wiki/Circuit_de_Monaco           |
|{USA, 39.795, Indianapolis, -86.2347}  |indianapolis|Indianapolis Motor Speedway |https://en.wikipedia.org/wiki/Indianapolis_Motor_Speedway |
|{Switzerland, 46.9589, Bern, 7.40194}  |bremgarten  |Circuit Bremgarten          |https://en.wikipedia.org/wiki/Circuit_Bremgarten          |

In [35]:
sesiones = ["FirstPractice", "SecondPractice", "ThirdPractice", "Qualifying",
            "Sprint", "SprintQualifying", "SprintShootout"]

total_filas = df_races_raw.count()

print(f"Total de filas (carreras): {total_filas}\n")

conteos_con_dato = [
    count(when(col(f"{s}.date").isNotNull(), True)).alias(s)
    for s in sesiones
]

fila = df_races_raw.select(conteos_con_dato).first()

print(f"{'Sesión':18s} | {'Con datos':>9s} | {'Vacías':>9s}")
print(f"{'-'*18}-+-{'-'*9}-+-{'-'*7}")
for s in sesiones:
    con_dato = fila[s]
    vacias   = total_filas - con_dato
    print(f"{s:18s} | {con_dato:9d} | {vacias:9d}")

Total de filas (carreras): 1171

Sesión             | Con datos |    Vacías
-------------------+-----------+--------
FirstPractice      |       421 |       750
SecondPractice     |       397 |       774
ThirdPractice      |       391 |       780
Qualifying         |       421 |       750
Sprint             |        30 |      1141
SprintQualifying   |        18 |      1153
SprintShootout     |         6 |      1165


In [37]:
nulos(df_races_raw)
pk_unica(df_races_raw, ["season", "round"])

Nulos / vacíos por columna:
+----+--------+-----+------+----+---+
|date|raceName|round|season|time|url|
+----+--------+-----+------+----+---+
|0   |0       |0    |0     |731 |0  |
+----+--------+-----+------+----+---+

PK ['season', 'round']: filas=1171, combinaciones distintas=1171, es única = True


In [44]:
# ------------------------------------------- RESULTS -------------------------------------------

df_results_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/results.json")
perfil(df_results_raw, "results (LND)", False)

===== Tabla: results (LND) =====
Filas: 1410   |   Columnas: 8
Nombres de columnas: ['Circuit', 'Results', 'date', 'raceName', 'round', 'season', 'time', 'url']

Esquema:
root
 |-- Circuit: struct (nullable = true)
 |    |-- Location: struct (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- lat: string (nullable = true)
 |    |    |-- locality: string (nullable = true)
 |    |    |-- long: string (nullable = true)
 |    |-- circuitId: string (nullable = true)
 |    |-- circuitName: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- Results: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Constructor: struct (nullable = true)
 |    |    |    |-- constructorId: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nationality: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |-- Driver: struct (nullable = true)
 |    |   

In [50]:
df_carreras = df_results_raw.select(
    "date",
    "raceName",
    "round",
    "season",
    col("Circuit.circuitId").alias("circuitId"),
    col("Circuit.circuitName").alias("circuitName"),
    col("Circuit.Location.country").alias("country"),
)
df_carreras.show(5, truncate=False)

+----------+------------------+-----+------+------------+----------------------------+-----------+
|date      |raceName          |round|season|circuitId   |circuitName                 |country    |
+----------+------------------+-----+------+------------+----------------------------+-----------+
|1950-05-13|British Grand Prix|1    |1950  |silverstone |Silverstone Circuit         |UK         |
|1950-05-21|Monaco Grand Prix |2    |1950  |monaco      |Circuit de Monaco           |Monaco     |
|1950-05-30|Indianapolis 500  |3    |1950  |indianapolis|Indianapolis Motor Speedway |USA        |
|1950-06-04|Swiss Grand Prix  |4    |1950  |bremgarten  |Circuit Bremgarten          |Switzerland|
|1950-06-18|Belgian Grand Prix|5    |1950  |spa         |Circuit de Spa-Francorchamps|Belgium    |
+----------+------------------+-----+------+------------+----------------------------+-----------+
only showing top 5 rows



In [52]:
df_resultados = df_results_raw.select(
    "season",
    "round",
    explode("Results").alias("res"),
)

In [53]:
df_resultados.select(
    "season", "round",
    col("res.Driver.driverId").alias("driverId"),
    col("res.Driver.givenName").alias("nombre"),
    col("res.Driver.familyName").alias("apellido"),
    col("res.Constructor.constructorId").alias("constructorId"),
    col("res.Constructor.name").alias("escuderia"),
).show(5, truncate=False)

+------+-----+-----------+------+---------+-------------+-----------+
|season|round|driverId   |nombre|apellido |constructorId|escuderia  |
+------+-----+-----------+------+---------+-------------+-----------+
|1950  |1    |farina     |Nino  |Farina   |alfa         |Alfa Romeo |
|1950  |1    |fagioli    |Luigi |Fagioli  |alfa         |Alfa Romeo |
|1950  |1    |reg_parnell|Reg   |Parnell  |alfa         |Alfa Romeo |
|1950  |1    |cabantous  |Yves  |Cabantous|lago         |Talbot-Lago|
|1950  |1    |rosier     |Louis |Rosier   |lago         |Talbot-Lago|
+------+-----+-----------+------+---------+-------------+-----------+
only showing top 5 rows



In [54]:
df_resultados.select(
    "season", "round",
    col("res.Driver.driverId").alias("driverId"),
    col("res.grid").alias("grid"),
    col("res.position").alias("position"),
    col("res.positionText").alias("positionText"),
    col("res.points").alias("points"),
    col("res.laps").alias("laps"),
    col("res.status").alias("status"),
).show(5, truncate=False)

+------+-----+-----------+----+--------+------------+------+----+--------+
|season|round|driverId   |grid|position|positionText|points|laps|status  |
+------+-----+-----------+----+--------+------------+------+----+--------+
|1950  |1    |farina     |1   |1       |1           |9     |70  |Finished|
|1950  |1    |fagioli    |2   |2       |2           |6     |70  |Finished|
|1950  |1    |reg_parnell|4   |3       |3           |4     |70  |Finished|
|1950  |1    |cabantous  |6   |4       |4           |3     |68  |+2 Laps |
|1950  |1    |rosier     |9   |5       |5           |2     |68  |+2 Laps |
+------+-----+-----------+----+--------+------------+------+----+--------+
only showing top 5 rows



In [55]:
df_resultados.select(
    "season", "round",
    col("res.Driver.driverId").alias("driverId"),
    col("res.Time.millis").alias("time_millis"),
    col("res.Time.time").alias("time"),
    col("res.FastestLap.lap").alias("fl_lap"),
    col("res.FastestLap.rank").alias("fl_rank"),
    col("res.FastestLap.Time.time").alias("fl_time"),
    col("res.FastestLap.AverageSpeed.speed").alias("fl_speed"),
).show(5, truncate=False)

+------+-----+-----------+-----------+-----------+------+-------+-------+--------+
|season|round|driverId   |time_millis|time       |fl_lap|fl_rank|fl_time|fl_speed|
+------+-----+-----------+-----------+-----------+------+-------+-------+--------+
|1950  |1    |farina     |8003600    |2:13:23.600|null  |null   |null   |null    |
|1950  |1    |fagioli    |8006200    |+2.600     |null  |null   |null   |null    |
|1950  |1    |reg_parnell|8055600    |+52.000    |null  |null   |null   |null    |
|1950  |1    |cabantous  |null       |null       |null  |null   |null   |null    |
|1950  |1    |rosier     |null       |null       |null  |null   |null   |null    |
+------+-----+-----------+-----------+-----------+------+-------+-------+--------+
only showing top 5 rows



In [56]:
df_res_flat = df_results_raw.select(explode("Results").alias("res")).select(
    # --- Constructor ---
    col("res.Constructor.constructorId").alias("constructor_id"),
    col("res.Constructor.name").alias("constructor_name"),
    col("res.Constructor.nationality").alias("constructor_nationality"),
    col("res.Constructor.url").alias("constructor_url"),
    # --- Driver ---
    col("res.Driver.code").alias("driver_code"),
    col("res.Driver.dateOfBirth").alias("driver_dateOfBirth"),
    col("res.Driver.driverId").alias("driver_id"),
    col("res.Driver.familyName").alias("driver_familyName"),
    col("res.Driver.givenName").alias("driver_givenName"),
    col("res.Driver.nationality").alias("driver_nationality"),
    col("res.Driver.permanentNumber").alias("driver_permanentNumber"),
    col("res.Driver.url").alias("driver_url"),
    # --- FastestLap ---
    col("res.FastestLap.AverageSpeed.speed").alias("fl_avgspeed_speed"),
    col("res.FastestLap.AverageSpeed.units").alias("fl_avgspeed_units"),
    col("res.FastestLap.Time.time").alias("fl_time"),
    col("res.FastestLap.lap").alias("fl_lap"),
    col("res.FastestLap.rank").alias("fl_rank"),
    # --- Time ---
    col("res.Time.millis").alias("time_millis"),
    col("res.Time.time").alias("time_time"),
    # --- escalares ---
    col("res.grid").alias("grid"),
    col("res.laps").alias("laps"),
    col("res.number").alias("number"),
    col("res.points").alias("points"),
    col("res.position").alias("position"),
    col("res.positionText").alias("positionText"),
    col("res.status").alias("status"),
)

In [97]:
reporte_nulos(df_res_flat, "resultados (piloto-carrera)")

Total de resultados (piloto-carrera): 26071

Columna                  |   Nulos |  Vacíos | Con dato | % nulos
------------------------------------------------------------------
constructor_id           |       0 |       0 |    26071 |    0.0%
constructor_name         |       0 |       0 |    26071 |    0.0%
constructor_nationality  |       0 |       0 |    26071 |    0.0%
constructor_url          |       0 |       0 |    26071 |    0.0%
driver_code              |   15246 |       0 |    10825 |   58.5%
driver_dateOfBirth       |       0 |       0 |    26071 |    0.0%
driver_id                |       0 |       0 |    26071 |    0.0%
driver_familyName        |       0 |       0 |    26071 |    0.0%
driver_givenName         |       0 |       0 |    26071 |    0.0%
driver_nationality       |       0 |       0 |    26071 |    0.0%
driver_permanentNumber   |   18833 |       0 |     7238 |   72.2%
driver_url               |       0 |       0 |    26071 |    0.0%
fl_avgspeed_speed        |   1

In [59]:
df_res_flat.show(5, truncate=False)

+--------------+----------------+-----------------------+-------------------------------------------------------+-----------+------------------+-----------+-----------------+----------------+------------------+----------------------+--------------------------------------------------+-----------------+-----------------+-------+------+-------+-----------+-----------+----+----+------+------+--------+------------+--------+
|constructor_id|constructor_name|constructor_nationality|constructor_url                                        |driver_code|driver_dateOfBirth|driver_id  |driver_familyName|driver_givenName|driver_nationality|driver_permanentNumber|driver_url                                        |fl_avgspeed_speed|fl_avgspeed_units|fl_time|fl_lap|fl_rank|time_millis|time_time  |grid|laps|number|points|position|positionText|status  |
+--------------+----------------+-----------------------+-------------------------------------------------------+-----------+------------------+----------

In [81]:
# buscamos la PK de results
df_res_pk = (df_results_raw
    .select("season", "round", explode("Results").alias("res"))
    .select("season", "round", col("res.Driver.driverId").alias("driverId"), col("res.number").alias("number")))

pk_unica(df_res_pk, ["season", "round", "driverId", "number"])

PK ['season', 'round', 'driverId', 'number']: filas=26071, combinaciones distintas=26071, es única = True


In [17]:
# ------------------------------------------- CONSTRUCTOR_STANDINGS -------------------------------------------

df_cs_raw = spark.read.option("multiLine", True).json(f"{LND_PATH}/constructor_standings.json")
perfil(df_cs_raw, "constructors standings (LND)", False)

===== Tabla: constructors standings (LND) =====
Filas: 69   |   Columnas: 3
Nombres de columnas: ['ConstructorStandings', 'round', 'season']

Esquema:
root
 |-- ConstructorStandings: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Constructor: struct (nullable = true)
 |    |    |    |-- constructorId: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nationality: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |-- points: string (nullable = true)
 |    |    |-- position: string (nullable = true)
 |    |    |-- positionText: string (nullable = true)
 |    |    |-- wins: string (nullable = true)
 |-- round: string (nullable = true)
 |-- season: string (nullable = true)



In [21]:
df_cs_flat = df_cs_raw.select(
    "season",
    "round",
    explode("ConstructorStandings").alias("cs"),
).select(
    "season",
    "round",
    col("cs.Constructor.constructorId").alias("constructor_id"),
    col("cs.Constructor.name").alias("constructor_name"),
    col("cs.Constructor.nationality").alias("constructor_nationality"),
    col("cs.Constructor.url").alias("constructor_url"),
    col("cs.points").alias("points"),
    col("cs.position").alias("position"),
    col("cs.positionText").alias("positionText"),
    col("cs.wins").alias("wins"),
)

df_cs_flat.show(5, truncate=True)

+------+-----+--------------+----------------+-----------------------+--------------------+------+--------+------------+----+
|season|round|constructor_id|constructor_name|constructor_nationality|     constructor_url|points|position|positionText|wins|
+------+-----+--------------+----------------+-----------------------+--------------------+------+--------+------------+----+
|  1958|   11|       vanwall|         Vanwall|                British|https://en.wikipe...|    48|       1|           1|   6|
|  1958|   11|       ferrari|         Ferrari|                Italian|https://en.wikipe...|    40|       2|           2|   2|
|  1958|   11|        cooper|          Cooper|                British|https://en.wikipe...|    31|       3|           3|   2|
|  1958|   11|           brm|             BRM|                British|https://en.wikipe...|    18|       4|           4|   0|
|  1958|   11|      maserati|        Maserati|                Italian|https://en.wikipe...|     6|       5|           

In [95]:
reporte_nulos(df_cs_flat,  "filas (escudería-snapshot)")

Total de filas (escudería-snapshot): 924

Columna                  |   Nulos |  Vacíos | Con dato | % nulos
------------------------------------------------------------------
season                   |       0 |       0 |      924 |    0.0%
round                    |       0 |       0 |      924 |    0.0%
constructor_id           |       0 |       0 |      924 |    0.0%
constructor_name         |       0 |       0 |      924 |    0.0%
constructor_nationality  |       0 |       0 |      924 |    0.0%
constructor_url          |       0 |       0 |      924 |    0.0%
points                   |       0 |       0 |      924 |    0.0%
position                 |     212 |       0 |      712 |   22.9%
positionText             |       0 |       0 |      924 |    0.0%
wins                     |       0 |       0 |      924 |    0.0%


In [22]:
# buscamos la PK de constructor standings
pk_unica(df_cs_flat, ["season", "constructor_id"])

PK ['season', 'constructor_id']: filas=924, combinaciones distintas=924, es única = True


In [10]:
df_cs_recorte = aplanar(DATAFRAMES["constructor_standings"]) \
    .filter((F.col("season") >= 1980) & (F.col("season") <= 1982))

df_cs_recorte.show(1000, truncate=False)

+----------------------------------------------+-------------------------------------+--------------------------------------------+-------------------------------------------------------------+---------------------------+-----------------------------+---------------------------------+-------------------------+-----+------+
|ConstructorStandings.Constructor.constructorId|ConstructorStandings.Constructor.name|ConstructorStandings.Constructor.nationality|ConstructorStandings.Constructor.url                         |ConstructorStandings.points|ConstructorStandings.position|ConstructorStandings.positionText|ConstructorStandings.wins|round|season|
+----------------------------------------------+-------------------------------------+--------------------------------------------+-------------------------------------------------------------+---------------------------+-----------------------------+---------------------------------+-------------------------+-----+------+
|williams                